In [1]:
# Import Libraries and Functions

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split, KFold

from sklearn.linear_model import LinearRegression

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_squared_error

from sklearn.utils import resample

In [6]:
df = pd.read_csv("APMM9-dataset07.csv")

In [7]:
df.head()

,X,y
0,0.846512,1069.348397
1,-0.747815,947.640172
2,-0.466257,989.091086
3,-1.026190,948.119486
4,1.063518,1083.872320


In [8]:
# Separate features and target
X = df.drop(columns=['y'])  # assuming the target is named 'y'
y = df['y']

In [9]:
# Define test sizes
test_sizes = [0.4, 0.3, 0.2, 0.1, 0.05]
rmse_list = []

In [14]:
for test_size in test_sizes:
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=2120
    )
    
    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Fit model
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    
    # Predict and evaluate
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    rmse_list.append(rmse)

# Average RMSE
avg_rmse = round(np.mean(rmse_list), 3)
print("Average RMSE over all splits:", avg_rmse)

std_mse = round(np.std(rmse_list), 3)
print("Standard Deviation of RMSE over all splits:", std_mse)

Average RMSE over all splits: 26.322
Standard Deviation of RMSE over all splits: 3.288


In [15]:
# -------------------------------
# 2. Average RMSE over 5-Fold CV
# -------------------------------
kf = KFold(n_splits=5, shuffle=False)
rmse_folds = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train model
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    rmse_folds.append(rmse)

avg_rmse_folds = round(np.mean(rmse_folds), 3)

# -------------------------------
# Results
# -------------------------------
print("Average RMSE over 5-Fold CV:", avg_rmse_folds)

Average RMSE over 5-Fold CV: 30.006


In [18]:
# -------------------------------
# Average RMSE over K-Fold CV (changeable)
# -------------------------------
num_folds = 10  # change to 5, 10, 20, etc.
kf = KFold(n_splits=num_folds, shuffle=False)
rmse_folds = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train model
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    rmse_folds.append(rmse)

avg_rmse_folds = round(np.mean(rmse_folds), 3)
print(f"Average RMSE over {num_folds}-Fold CV:", avg_rmse_folds)


Average RMSE over 10-Fold CV: 29.95


In [20]:
from sklearn.model_selection import LeaveOneOut

In [21]:
# Set up LOOCV
loo = LeaveOneOut()
rmse_list = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train model
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    rmse_list.append(rmse)

# Average RMSE over LOOCV
avg_rmse_loocv = round(np.mean(rmse_list), 3)
print("Average RMSE over LOOCV:", avg_rmse_loocv)

Average RMSE over LOOCV: 24.337


In [24]:
# Bootstrapping
num_iterations = 100
rmse_list = []

n_samples = len(data)

for boot in range(num_iterations):
    # Sample with replacement
    boot_sample = resample(data, replace=True, n_samples=n_samples, random_state=10 * boot)
    
    # Get OOB samples
    boot_indices = boot_sample.index
    oob_indices = list(set(data.index) - set(boot_indices))
    
    if len(oob_indices) == 0:
        continue  # skip if no OOB data (rare, but possible)
    
    oob_data = data.loc[oob_indices]
    
    # Separate features and target
    X_train = boot_sample.drop(columns=['y'])
    y_train = boot_sample['y']
    X_test = oob_data.drop(columns=['y'])
    y_test = oob_data['y']
    
    # Train and predict
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Compute RMSE
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    rmse_list.append(rmse)

# Final average RMSE over all bootstrap OOB test sets
avg_rmse_bootstrap = round(np.mean(rmse_list), 3)
print("Average RMSE over 100 bootstraps (OOB testing):", avg_rmse_bootstrap)


Average RMSE over 100 bootstraps (OOB testing): 30.105


In [26]:
# Bootstrapping
num_iterations = 200
rmse_list = []

n_samples = len(data)

for boot in range(num_iterations):
    # Sample with replacement
    boot_sample = resample(data, replace=True, n_samples=n_samples, random_state=10 * boot)
    
    # Get OOB samples
    boot_indices = boot_sample.index
    oob_indices = list(set(data.index) - set(boot_indices))
    
    if len(oob_indices) == 0:
        continue  # skip if no OOB data (rare, but possible)
    
    oob_data = data.loc[oob_indices]
    
    # Separate features and target
    X_train = boot_sample.drop(columns=['y'])
    y_train = boot_sample['y']
    X_test = oob_data.drop(columns=['y'])
    y_test = oob_data['y']
    
    # Train and predict
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Compute RMSE
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    rmse_list.append(rmse)

# Final average RMSE over all bootstrap OOB test sets
avg_rmse_bootstrap = round(np.mean(rmse_list), 3)
print("Average RMSE over 200 bootstraps (OOB testing):", avg_rmse_bootstrap)


Average RMSE over 200 bootstraps (OOB testing): 30.13


In [27]:
# Bootstrapping
num_iterations = 400
rmse_list = []

n_samples = len(data)

for boot in range(num_iterations):
    # Sample with replacement
    boot_sample = resample(data, replace=True, n_samples=n_samples, random_state=10 * boot)
    
    # Get OOB samples
    boot_indices = boot_sample.index
    oob_indices = list(set(data.index) - set(boot_indices))
    
    if len(oob_indices) == 0:
        continue  # skip if no OOB data (rare, but possible)
    
    oob_data = data.loc[oob_indices]
    
    # Separate features and target
    X_train = boot_sample.drop(columns=['y'])
    y_train = boot_sample['y']
    X_test = oob_data.drop(columns=['y'])
    y_test = oob_data['y']
    
    # Train and predict
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Compute RMSE
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    rmse_list.append(rmse)

# Final average RMSE over all bootstrap OOB test sets
avg_rmse_bootstrap = round(np.mean(rmse_list), 3)
print("Average RMSE over 400 bootstraps (OOB testing):", avg_rmse_bootstrap)


Average RMSE over 400 bootstraps (OOB testing): 30.059
